<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_setfit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Task SetFit for VeriPromiseESG

This notebook is a standalone SetFit-style training and inference flow. It does not modify the original `model_train.ipynb`, `model_inference.ipynb`, `compare_result.py`, or data files.

Design choices:
- Keep the original fold CSV input format and final submission CSV output format.
- Keep the original hierarchical routing: T1 controls T2-T4, and T3 controls T4.
- Use one shared SentenceTransformer encoder for T1-T4 contrastive fine-tuning, then train four lightweight task heads on the shared embedding space.
- Use a Chinese sentence embedding model by default: `BAAI/bge-large-zh-v1.5`, with `BAAI/bge-base-zh-v1.5` as fallback.

References:
- SetFit docs: https://huggingface.co/docs/setfit/index
- BGE zh model: https://huggingface.co/BAAI/bge-large-zh-v1.5
- SetFit paper: https://arxiv.org/abs/2209.11055


In [28]:
# Install dependencies. Restart the runtime/kernel after this cell if your environment asks for it.
# !pip install -q setfit sentence-transformers datasets accelerate transformers scikit-learn pandas numpy tqdm joblib huggingface_hub


In [29]:
from huggingface_hub import HfApi, login, create_repo
import os
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import gc
import json
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory

from datasets import Dataset
import joblib
import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi, notebook_login
from sentence_transformers import (
    InputExample,
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import BinaryClassificationEvaluator, SentenceEvaluator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
from transformers import EarlyStoppingCallback

try:
    import setfit
except Exception:
    setfit = None

warnings.filterwarnings("ignore")


In [ ]:
# ==========================================
# 0. Configuration
# ==========================================

RAW_BASE_URL = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/feat-model-train/app/data/clean_data/"
USE_GITHUB_RAW = True

# Default mirrors the original inference notebook, which uses val_fold_1 as a test target.
# Change this to the actual test CSV when producing a challenge submission.
TEST_CSV_PATH = f"{RAW_BASE_URL}val_fold_1.csv"

FOLDS = [1, 2, 3, 4, 5]
SEED = 42
if len(FOLDS) > 5 or len(set(FOLDS)) != len(FOLDS):
    raise ValueError("This notebook permits at most five unique encoder-training folds.")

PRIMARY_MODEL_NAME = "BAAI/bge-base-zh-v1.5"
FALLBACK_MODEL_NAME = "BAAI/bge-base-zh-v1.5"
FORCE_FALLBACK_MODEL = False

MAX_SEQ_LENGTH = 512
HEAD_RATIO = 0.25

# One fixed pair profile. No encoder-profile search is performed.
POSITIVE_PAIR_TARGETS = {
    "t1": {"T1::No": 500, "T1::Yes": 500},
    "t2": {
        "T2::already": 500,
        "T2::within_2_years": 378,
        "T2::between_2_and_5_years": 500,
        "T2::longer_than_5_years": 500,
    },
    "t3": {"T3::No": 500, "T3::Yes": 500},
    "t4": {"T4::Clear": 600, "T4::Not Clear": 600, "T4::Misleading": 300},
}
NEGATIVE_PAIRS_PER_LABEL = {"t1": 500, "t2": 500, "t3": 500, "t4": 500}
HARD_NEGATIVE_PAIR_TARGETS = {
    "t1": [("T1::No", "T1::Yes", 240)],
    "t2": [
        ("T2::already", "T2::within_2_years", 200),
        ("T2::within_2_years", "T2::between_2_and_5_years", 200),
        ("T2::between_2_and_5_years", "T2::longer_than_5_years", 200),
    ],
    "t3": [("T3::No", "T3::Yes", 240)],
    "t4": [
        ("T4::Clear", "T4::Not Clear", 480),
        ("T4::Clear", "T4::Misleading", 160),
        ("T4::Not Clear", "T4::Misleading", 160),
    ],
}
T2_LABEL_ORDER = [
    "T2::already",
    "T2::within_2_years",
    "T2::between_2_and_5_years",
    "T2::longer_than_5_years",
]
VALIDATION_PAIRS_PER_LABEL = 120

CONTRASTIVE_EPOCHS = 8
CONTRASTIVE_BATCH_SIZE = 16
CONTRASTIVE_LEARNING_RATE = 2e-5
CONTRASTIVE_WARMUP_RATIO = 0.1
CONTRASTIVE_WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0001

HEAD_MAX_ITER = 3000
HEAD_C = 1.0
SYNTHETIC_ID_MIN = 90000
SYNTHETIC_T4_SAMPLE_WEIGHT = 0.35

REFERENCE_OOF_MACRO_F1 = {
    "promise_status": 0.766654,
    "evidence_status": 0.709650,
}
T1_T3_MAX_ALLOWED_DROP = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("setfit_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OOF_PREDICTION_CSV = OUTPUT_DIR / "setfit_oof_predictions.csv"
OOF_PROBABILITY_CSV = OUTPUT_DIR / "setfit_oof_probabilities.csv"
THRESHOLD_JSON = OUTPUT_DIR / "setfit_thresholds.json"
INFERENCE_CONFIG_JSON = OUTPUT_DIR / "setfit_inference_config.json"
SUBMISSION_CSV = Path("setfit_final_submission.csv")

# Hugging Face Hub upload settings. Leave HF_SETFIT_REPO_ID=None to use
# {your_hf_username}/VeriPromise_ESG_2026_9906_SetFit after notebook_login().
RUN_HF_UPLOAD = True
HF_SETFIT_REPO_ID = None
HF_PRIVATE_REPO = False
HF_COMMIT_MESSAGE = "Upload Multi-Task SetFit artifacts"

print(f"Device: {DEVICE}")
print(f"SetFit package: {getattr(setfit, '__version__', 'not imported')}")


In [ ]:
# ==========================================
# 1. Task definitions and helpers
# ==========================================

ID_COLUMN = "id"
TEXT_COLUMN = "data"

TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

MISSING_ALLOWED_COLUMNS = {
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
}

TASK_PREFIX = {
    "t1": "任務：判斷是否有 ESG 承諾。",
    "t2": "任務：判斷承諾驗證時間。",
    "t3": "任務：判斷是否提供證據。",
    "t4": "任務：判斷證據品質。",
}

TASK_COLUMN = {
    "t1": "promise_status",
    "t2": "verification_timeline",
    "t3": "evidence_status",
    "t4": "evidence_quality",
}

TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}

COMPETITION_SCORE_WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15,
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "app" / "data" / "clean_data").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
LOCAL_CLEAN_DATA_DIR = PROJECT_ROOT / "app" / "data" / "clean_data"
LOCAL_FINAL_SUBMISSION = PROJECT_ROOT / "final_submission.csv"


def read_fold_csv(fold_num, split):
    filename = f"{split}_fold_{fold_num}.csv"
    if USE_GITHUB_RAW:
        try:
            return pd.read_csv(f"{RAW_BASE_URL}{filename}")
        except Exception as exc:
            print(f"GitHub raw read failed for {filename}: {exc}. Falling back to local clean_data.")
    return pd.read_csv(LOCAL_CLEAN_DATA_DIR / filename)


def normalize_value(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    if value == "" or value.upper() == "N/A":
        return None
    if value == "more_than_5_years":
        return "longer_than_5_years"
    return value


def normalize_series(series):
    return series.apply(normalize_value)


def is_synthetic_row(row):
    try:
        return int(row[ID_COLUMN]) >= SYNTHETIC_ID_MIN
    except Exception:
        return False


def truncate_text_by_tokens(text, tokenizer, max_seq_length=MAX_SEQ_LENGTH, head_ratio=HEAD_RATIO):
    """Head-tail truncation without calling tokenizer.encode on overlong text.

    Some Hugging Face tokenizers warn or error when encode() sees >512 tokens before
    truncation. tokenize() lets us count and trim first, then reconstruct safe text.
    """
    text = str(text)
    max_body_len = max_seq_length - 2
    tokens = tokenizer.tokenize(text)
    if len(tokens) <= max_body_len:
        return text

    head_len = int(max_body_len * head_ratio)
    tail_len = max_body_len - head_len
    kept_tokens = tokens[:head_len] + tokens[-tail_len:]
    if hasattr(tokenizer, "convert_tokens_to_string"):
        return tokenizer.convert_tokens_to_string(kept_tokens)

    kept_ids = tokenizer.convert_tokens_to_ids(kept_tokens)
    return tokenizer.decode(kept_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)


def build_base_text(row, tokenizer=None):
    raw_text = str(row.get(TEXT_COLUMN, ""))
    full_text = f"文本：{raw_text}"
    if tokenizer is not None:
        full_text = truncate_text_by_tokens(full_text, tokenizer)
    return full_text


def build_task_text(task_key, row, tokenizer=None):
    return TASK_PREFIX[task_key] + build_base_text(row, tokenizer=tokenizer)


def validate_required_columns(df, name):
    required = [ID_COLUMN, TEXT_COLUMN] + TARGET_COLUMNS
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


In [ ]:
# ==========================================
# 2. Multi-task SetFit data construction
# ==========================================

def filter_task_dataframe(df, task_key, include_synthetic_for_t4=True):
    work = df.copy()
    for col in TARGET_COLUMNS:
        if col in work.columns:
            work[col] = normalize_series(work[col])

    if task_key != "t4" or not include_synthetic_for_t4:
        work = work[~work.apply(is_synthetic_row, axis=1)]

    if task_key == "t1":
        mask = work["promise_status"].notna()
    elif task_key == "t2":
        mask = (work["promise_status"] == "Yes") & work["verification_timeline"].notna()
    elif task_key == "t3":
        mask = (work["promise_status"] == "Yes") & work["evidence_status"].notna()
    elif task_key == "t4":
        mask = (
            (work["promise_status"] == "Yes")
            & (work["evidence_status"] == "Yes")
            & work["evidence_quality"].notna()
        )
    else:
        raise ValueError(f"Unknown task: {task_key}")

    filtered = work.loc[mask].copy()
    filtered = filtered[filtered[TASK_COLUMN[task_key]].isin(TASK_CLASSES[task_key])]
    return filtered.reset_index(drop=True)


def build_multitask_views(train_df, tokenizer, include_synthetic_for_t4=True):
    rows = []
    for task_key in ["t1", "t2", "t3", "t4"]:
        task_df = filter_task_dataframe(
            train_df,
            task_key,
            include_synthetic_for_t4=include_synthetic_for_t4,
        )
        label_col = TASK_COLUMN[task_key]
        for _, row in task_df.iterrows():
            label = normalize_value(row[label_col])
            rows.append(
                {
                    "id": row[ID_COLUMN],
                    "task": task_key,
                    "text": build_task_text(task_key, row, tokenizer=tokenizer),
                    "label": f"{task_key.upper()}::{label}",
                    "is_synthetic": is_synthetic_row(row),
                }
            )
    views = pd.DataFrame(rows)
    if views.empty:
        raise ValueError("No training views were created.")
    return views


def canonical_pair_key(text_a, text_b):
    if text_a == text_b:
        raise ValueError("Self-pairs are not allowed.")
    return tuple(sorted((text_a, text_b)))


def sample_unique_positive_pairs(texts, target_count, rng):
    texts = list(dict.fromkeys(texts))
    available = len(texts) * (len(texts) - 1) // 2
    target_count = min(int(target_count), available)
    if target_count <= 0:
        return []

    if target_count == available:
        return [
            (texts[left], texts[right])
            for left in range(len(texts))
            for right in range(left + 1, len(texts))
        ]

    selected_indices = set()
    while len(selected_indices) < target_count:
        left, right = sorted(rng.choice(len(texts), size=2, replace=False).tolist())
        selected_indices.add((left, right))
    return [(texts[left], texts[right]) for left, right in selected_indices]


def t2_negative_weight(label_a, label_b):
    if label_a not in T2_LABEL_ORDER or label_b not in T2_LABEL_ORDER:
        return 1.0
    distance = abs(T2_LABEL_ORDER.index(label_a) - T2_LABEL_ORDER.index(label_b))
    return {1: 3.0, 2: 2.0, 3: 1.0}.get(distance, 1.0)


def sample_unique_negative_pairs(groups, anchor_label, target_count, rng, task_key, seen_pairs):
    anchor_texts = groups.get(anchor_label, [])
    other_labels = [label for label, texts in groups.items() if label != anchor_label and texts]
    if not anchor_texts or not other_labels:
        return []

    weights = np.array(
        [t2_negative_weight(anchor_label, label) if task_key == "t2" else 1.0 for label in other_labels],
        dtype=float,
    )
    weights /= weights.sum()
    selected = []
    local_seen = set()
    max_available = sum(len(anchor_texts) * len(groups[label]) for label in other_labels)
    target_count = min(int(target_count), max_available)
    max_attempts = max(10_000, target_count * 100)

    for _ in range(max_attempts):
        if len(selected) >= target_count:
            break
        other_label = rng.choice(other_labels, p=weights)
        text_a = rng.choice(anchor_texts)
        text_b = rng.choice(groups[other_label])
        pair_key = canonical_pair_key(text_a, text_b)
        if pair_key in seen_pairs or pair_key in local_seen:
            continue
        local_seen.add(pair_key)
        selected.append((text_a, text_b))

    if len(selected) != target_count:
        raise RuntimeError(
            f"Could only sample {len(selected)}/{target_count} unique negatives for {task_key} {anchor_label}."
        )
    seen_pairs.update(local_seen)
    return selected


def sample_pairs_for_task(task_views, rng, task_key):
    examples = []
    labels = sorted(task_views["label"].unique())
    groups = {
        label: task_views.loc[task_views["label"] == label, "text"].drop_duplicates().tolist()
        for label in labels
    }
    seen_pairs = set()
    positive_plan = {}
    negative_plan = {}

    for label in labels:
        target = POSITIVE_PAIR_TARGETS[task_key].get(label, 0)
        positive_pairs = sample_unique_positive_pairs(groups[label], target, rng)
        positive_plan[label] = len(positive_pairs)
        for text_a, text_b in positive_pairs:
            pair_key = canonical_pair_key(text_a, text_b)
            if pair_key in seen_pairs:
                raise AssertionError(f"Duplicate positive pair for {task_key}: {pair_key}")
            seen_pairs.add(pair_key)
            examples.append(InputExample(texts=[text_a, text_b], label=1.0))

    for label in labels:
        negative_pairs = sample_unique_negative_pairs(
            groups,
            anchor_label=label,
            target_count=NEGATIVE_PAIRS_PER_LABEL[task_key],
            rng=rng,
            task_key=task_key,
            seen_pairs=seen_pairs,
        )
        negative_plan[label] = len(negative_pairs)
        examples.extend(
            InputExample(texts=[text_a, text_b], label=0.0)
            for text_a, text_b in negative_pairs
        )

    print(f"{task_key} positive pair plan:", positive_plan)
    print(f"{task_key} negative pair plan:", negative_plan)
    return examples, seen_pairs


def mine_hard_pairs_for_label_pair(groups, embeddings_by_text, label_a, label_b, target_count, seen_pairs):
    texts_a = groups.get(label_a, [])
    texts_b = groups.get(label_b, [])
    if not texts_a or not texts_b or target_count <= 0:
        return []

    embeddings_a = np.vstack([embeddings_by_text[text] for text in texts_a])
    embeddings_b = np.vstack([embeddings_by_text[text] for text in texts_b])
    similarities = embeddings_a @ embeddings_b.T
    ranked_flat_indices = np.argsort(similarities, axis=None)[::-1]

    selected = []
    for flat_index in ranked_flat_indices:
        index_a, index_b = np.unravel_index(flat_index, similarities.shape)
        text_a = texts_a[index_a]
        text_b = texts_b[index_b]
        pair_key = canonical_pair_key(text_a, text_b)
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        selected.append((text_a, text_b))
        if len(selected) >= target_count:
            break
    return selected


def build_hard_negative_examples(task_views, encoder, task_key, seen_pairs):
    targets = HARD_NEGATIVE_PAIR_TARGETS.get(task_key, [])
    if encoder is None or not targets or task_views.empty:
        return []

    groups = {
        label: task_views.loc[task_views["label"] == label, "text"].drop_duplicates().tolist()
        for label in sorted(task_views["label"].unique())
    }
    texts = list(dict.fromkeys(task_views["text"].tolist()))
    embeddings = encoder.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    embeddings_by_text = dict(zip(texts, embeddings))

    hard_examples = []
    hard_plan = {}
    for label_a, label_b, target_count in targets:
        pairs = mine_hard_pairs_for_label_pair(
            groups,
            embeddings_by_text,
            label_a,
            label_b,
            target_count,
            seen_pairs,
        )
        hard_plan[f"{label_a}<->{label_b}"] = len(pairs)
        hard_examples.extend(
            InputExample(texts=[text_a, text_b], label=0.0)
            for text_a, text_b in pairs
        )
    print(f"{task_key} hard-negative plan:", hard_plan)
    return hard_examples


def assert_unique_examples(examples, task_key):
    pair_keys = [canonical_pair_key(example.texts[0], example.texts[1]) for example in examples]
    if len(pair_keys) != len(set(pair_keys)):
        raise AssertionError(f"{task_key} contains duplicate contrastive pairs.")


def build_contrastive_examples(multitask_views, encoder=None, seed=SEED):
    rng = np.random.default_rng(seed)
    all_examples = []
    for task_key in ["t1", "t2", "t3", "t4"]:
        task_views = multitask_views[multitask_views["task"] == task_key]
        task_examples, seen_pairs = sample_pairs_for_task(task_views, rng, task_key)
        task_examples.extend(
            build_hard_negative_examples(
                task_views,
                encoder=encoder,
                task_key=task_key,
                seen_pairs=seen_pairs,
            )
        )
        assert_unique_examples(task_examples, task_key)
        all_examples.extend(task_examples)
    rng.shuffle(all_examples)
    return all_examples


def build_validation_examples(val_df, tokenizer, seed=SEED):
    """Build fixed, task-grouped validation pairs from real data only."""
    rng = np.random.default_rng(seed)
    validation_views = build_multitask_views(
        val_df,
        tokenizer=tokenizer,
        include_synthetic_for_t4=False,
    )
    examples_by_task = {}

    for task_key in ["t1", "t2", "t3", "t4"]:
        task_views = validation_views[validation_views["task"] == task_key]
        labels = sorted(task_views["label"].unique())
        groups = {
            label: task_views.loc[task_views["label"] == label, "text"].drop_duplicates().tolist()
            for label in labels
        }
        task_examples = []
        seen_pairs = set()

        for label in labels:
            positive_pairs = sample_unique_positive_pairs(
                groups[label],
                VALIDATION_PAIRS_PER_LABEL,
                rng,
            )
            for text_a, text_b in positive_pairs:
                seen_pairs.add(canonical_pair_key(text_a, text_b))
                task_examples.append(InputExample(texts=[text_a, text_b], label=1.0))

        for label in labels:
            negative_pairs = sample_unique_negative_pairs(
                groups,
                anchor_label=label,
                target_count=VALIDATION_PAIRS_PER_LABEL,
                rng=rng,
                task_key=task_key,
                seen_pairs=seen_pairs,
            )
            task_examples.extend(
                InputExample(texts=[text_a, text_b], label=0.0)
                for text_a, text_b in negative_pairs
            )

        assert_unique_examples(task_examples, f"{task_key} validation")
        if not task_examples:
            raise ValueError(f"No validation pairs were created for {task_key}.")
        examples_by_task[task_key] = task_examples

    examples = [
        example
        for task_key in ["t1", "t2", "t3", "t4"]
        for example in examples_by_task[task_key]
    ]
    rng.shuffle(examples)
    return examples, examples_by_task


def examples_to_dataset(examples):
    return Dataset.from_dict(
        {
            "sentence1": [example.texts[0] for example in examples],
            "sentence2": [example.texts[1] for example in examples],
            "label": [float(example.label) for example in examples],
        }
    )


In [ ]:
# ==========================================
# 3. Model training: shared SetFit encoder + 4 heads
# ==========================================

class WeightedTaskPairEvaluator(SentenceEvaluator):
    """Return an explicit four-task weighted AP for checkpoint selection."""

    def __init__(self, examples_by_task, task_weights, batch_size=16):
        super().__init__()
        self.primary_metric = "weighted_ap"
        self.task_weights = dict(task_weights)
        self.evaluators = {
            task_key: BinaryClassificationEvaluator(
                sentences1=[example.texts[0] for example in examples],
                sentences2=[example.texts[1] for example in examples],
                labels=[int(example.label) for example in examples],
                name="",
                batch_size=batch_size,
                show_progress_bar=False,
                write_csv=False,
                similarity_fn_names=["cosine"],
            )
            for task_key, examples in examples_by_task.items()
        }

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        metrics = {}
        weighted_sum = 0.0
        used_weight = 0.0
        for task_key in ["t1", "t2", "t3", "t4"]:
            evaluator = self.evaluators[task_key]
            task_metrics = evaluator(model, output_path=None, epoch=epoch, steps=steps)
            task_ap = float(task_metrics[evaluator.primary_metric])
            metrics[f"{task_key}_ap"] = task_ap
            weight = float(self.task_weights[task_key])
            weighted_sum += weight * task_ap
            used_weight += weight
        metrics[self.primary_metric] = weighted_sum / used_weight
        return metrics


def sqrt_inverse_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    return {
        int(class_id): float(min(5.0, np.sqrt(max_count / count)))
        for class_id, count in zip(classes, counts)
    }


class OrdinalLogisticHead:
    """Three cumulative binary logistic models for the four ordered T2 classes."""

    def __init__(
        self,
        ordered_encoded_classes,
        C=1.0,
        class_weight_mode="balanced",
        max_iter=3000,
        random_state=42,
    ):
        self.ordered_encoded_classes = np.asarray(ordered_encoded_classes, dtype=int)
        self.C = C
        self.class_weight_mode = class_weight_mode
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        y = np.asarray(y, dtype=int)
        self.classes_ = np.sort(np.unique(y))
        if set(self.classes_) != set(self.ordered_encoded_classes):
            raise ValueError("Ordinal head requires all configured T2 classes.")

        encoded_to_rank = {
            int(encoded_class): rank
            for rank, encoded_class in enumerate(self.ordered_encoded_classes)
        }
        ranks = np.asarray([encoded_to_rank[int(value)] for value in y], dtype=int)
        self.models_ = []
        for threshold in range(len(self.ordered_encoded_classes) - 1):
            binary_y = (ranks > threshold).astype(int)
            if self.class_weight_mode == "balanced":
                class_weight = "balanced"
            elif self.class_weight_mode == "sqrt":
                class_weight = sqrt_inverse_class_weight(binary_y)
            else:
                class_weight = None
            model = LogisticRegression(
                C=self.C,
                class_weight=class_weight,
                max_iter=self.max_iter,
                random_state=self.random_state,
            )
            model.fit(X, binary_y, sample_weight=sample_weight)
            self.models_.append(model)
        return self

    def predict_proba(self, X):
        cumulative = np.column_stack(
            [model.predict_proba(X)[:, 1] for model in self.models_]
        )
        cumulative = np.minimum.accumulate(cumulative, axis=1)
        rank_probs = np.column_stack(
            [
                1.0 - cumulative[:, 0],
                cumulative[:, 0] - cumulative[:, 1],
                cumulative[:, 1] - cumulative[:, 2],
                cumulative[:, 2],
            ]
        )
        rank_probs = np.clip(rank_probs, 0.0, 1.0)
        rank_probs /= np.maximum(rank_probs.sum(axis=1, keepdims=True), 1e-12)

        encoded_probs = np.zeros((len(rank_probs), len(self.classes_)), dtype=float)
        for rank, encoded_class in enumerate(self.ordered_encoded_classes):
            encoded_column = int(np.where(self.classes_ == encoded_class)[0][0])
            encoded_probs[:, encoded_column] = rank_probs[:, rank]
        return encoded_probs

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def load_encoder():
    model_name = FALLBACK_MODEL_NAME if FORCE_FALLBACK_MODEL else PRIMARY_MODEL_NAME
    try:
        encoder = SentenceTransformer(model_name, device=DEVICE)
    except RuntimeError as exc:
        if model_name == PRIMARY_MODEL_NAME:
            print(f"Primary model failed to load: {exc}")
            print(f"Falling back to {FALLBACK_MODEL_NAME}")
            encoder = SentenceTransformer(FALLBACK_MODEL_NAME, device=DEVICE)
        else:
            raise
    encoder.max_seq_length = MAX_SEQ_LENGTH
    return encoder


def metric_from_log_entry(entry, suffix):
    for key, value in entry.items():
        if key.endswith(suffix):
            return float(value)
    return None


def print_encoder_training_summary(trainer, evaluator, fold):
    epochs = {}
    for entry in trainer.state.log_history:
        epoch = entry.get("epoch")
        if epoch is None:
            continue
        epoch_key = round(float(epoch), 4)
        row = epochs.setdefault(epoch_key, {"epoch": epoch_key})
        if "loss" in entry:
            row["training_loss"] = float(entry["loss"])
        if "eval_loss" in entry:
            row["validation_loss"] = float(entry["eval_loss"])
        weighted_ap = metric_from_log_entry(entry, evaluator.primary_metric)
        if weighted_ap is not None:
            row["weighted_ap"] = weighted_ap
        for task_key in ["t1", "t2", "t3", "t4"]:
            task_ap = metric_from_log_entry(entry, f"{task_key}_ap")
            if task_ap is not None:
                row[f"{task_key}_ap"] = task_ap

    history = pd.DataFrame(sorted(epochs.values(), key=lambda item: item["epoch"]))
    if not history.empty:
        print(f"Fold {fold} encoder training history:")
        print(history.to_string(index=False))

    if "weighted_ap" not in history.columns or not history["weighted_ap"].notna().any():
        raise RuntimeError("weighted_ap was not logged; checkpoint selection is unsafe.")
    best_row = history.loc[history["weighted_ap"].idxmax()]
    expected_best = float(best_row["weighted_ap"])
    actual_best = float(trainer.state.best_metric)
    if not np.isclose(actual_best, expected_best, rtol=1e-6, atol=1e-8):
        raise RuntimeError(
            f"Checkpoint metric mismatch: trainer={actual_best}, max weighted_ap={expected_best}."
        )

    stopped_epoch = float(trainer.state.epoch or 0.0)
    stopped_early = stopped_epoch + 1e-9 < float(CONTRASTIVE_EPOCHS)
    print(
        f"Fold {fold}: best_epoch={float(best_row['epoch'])}, "
        f"best_validation_weighted_ap={actual_best}, stopped_epoch={stopped_epoch}, "
        f"early_stopped={stopped_early}"
    )


def fine_tune_shared_encoder(encoder, train_df, val_df, fold, seed=SEED):
    multitask_views = build_multitask_views(train_df, tokenizer=encoder.tokenizer)
    examples = build_contrastive_examples(multitask_views, encoder=encoder, seed=seed + fold)
    validation_examples, validation_examples_by_task = build_validation_examples(
        val_df,
        tokenizer=encoder.tokenizer,
        seed=seed + 10_000 + fold,
    )
    print(
        f"Fold {fold}: multitask views={len(multitask_views)}, "
        f"contrastive pairs={len(examples)}, validation pairs={len(validation_examples)}"
    )
    print(multitask_views.groupby(["task", "label"]).size())

    train_dataset = examples_to_dataset(examples)
    validation_dataset = examples_to_dataset(validation_examples)
    evaluator = WeightedTaskPairEvaluator(
        validation_examples_by_task,
        task_weights={
            "t1": COMPETITION_SCORE_WEIGHTS["promise_status"],
            "t2": COMPETITION_SCORE_WEIGHTS["verification_timeline"],
            "t3": COMPETITION_SCORE_WEIGHTS["evidence_status"],
            "t4": COMPETITION_SCORE_WEIGHTS["evidence_quality"],
        },
        batch_size=CONTRASTIVE_BATCH_SIZE,
    )
    train_loss = losses.CosineSimilarityLoss(encoder)

    steps_per_epoch = int(np.ceil(len(train_dataset) / CONTRASTIVE_BATCH_SIZE))
    warmup_steps = max(
        1,
        int(steps_per_epoch * CONTRASTIVE_EPOCHS * CONTRASTIVE_WARMUP_RATIO),
    )
    with TemporaryDirectory(prefix=f"encoder_training_fold_{fold}_", dir=str(OUTPUT_DIR)) as checkpoint_dir:
        training_args = SentenceTransformerTrainingArguments(
            output_dir=checkpoint_dir,
            num_train_epochs=CONTRASTIVE_EPOCHS,
            per_device_train_batch_size=CONTRASTIVE_BATCH_SIZE,
            per_device_eval_batch_size=CONTRASTIVE_BATCH_SIZE,
            learning_rate=CONTRASTIVE_LEARNING_RATE,
            warmup_steps=warmup_steps,
            weight_decay=CONTRASTIVE_WEIGHT_DECAY,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model=evaluator.primary_metric,
            greater_is_better=True,
            save_total_limit=1,
            report_to="none",
            seed=seed + fold,
            data_seed=seed + fold,
        )
        trainer = SentenceTransformerTrainer(
            model=encoder,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            loss=train_loss,
            evaluator=evaluator,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=EARLY_STOPPING_PATIENCE,
                    early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
                )
            ],
        )
        trainer.train()
        print_encoder_training_summary(trainer, evaluator, fold)
        encoder = trainer.model

    encoder.max_seq_length = MAX_SEQ_LENGTH
    return encoder


def encode_task_texts(encoder, df, task_key, batch_size=32):
    texts = [build_task_text(task_key, row, tokenizer=encoder.tokenizer) for _, row in df.iterrows()]
    return encoder.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def build_calibrated_linear_svc(class_weight):
    base = LinearSVC(
        C=0.5,
        class_weight=class_weight,
        max_iter=5000,
        random_state=SEED,
    )
    try:
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=3)


def build_head_candidates(task_key, y, label_encoder):
    sqrt_weights = sqrt_inverse_class_weight(y)
    candidates = []
    for weight_name, class_weight in [
        ("balanced", "balanced"),
        ("sqrt", sqrt_weights),
    ]:
        candidates.extend(
            [
                (
                    f"logreg_{weight_name}",
                    LogisticRegression(
                        C=HEAD_C,
                        class_weight=class_weight,
                        max_iter=HEAD_MAX_ITER,
                        random_state=SEED,
                    ),
                ),
                (
                    f"linear_svc_calibrated_{weight_name}",
                    build_calibrated_linear_svc(class_weight),
                ),
            ]
        )

    if task_key == "t2":
        ordered_encoded_classes = label_encoder.transform(TASK_CLASSES["t2"]).tolist()
        candidates.extend(
            [
                (
                    "ordinal_logreg_balanced",
                    OrdinalLogisticHead(
                        ordered_encoded_classes=ordered_encoded_classes,
                        C=HEAD_C,
                        class_weight_mode="balanced",
                        max_iter=HEAD_MAX_ITER,
                        random_state=SEED,
                    ),
                ),
                (
                    "ordinal_logreg_sqrt",
                    OrdinalLogisticHead(
                        ordered_encoded_classes=ordered_encoded_classes,
                        C=HEAD_C,
                        class_weight_mode="sqrt",
                        max_iter=HEAD_MAX_ITER,
                        random_state=SEED,
                    ),
                ),
            ]
        )

    if task_key == "t4":
        candidates.append(
            (
                "mlp_256_64",
                MLPClassifier(
                    hidden_layer_sizes=(256, 64),
                    activation="relu",
                    alpha=1e-4,
                    batch_size=32,
                    learning_rate_init=1e-3,
                    max_iter=600,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=20,
                    random_state=SEED,
                ),
            )
        )
    return candidates


def fit_with_optional_sample_weight(classifier, embeddings, y, sample_weight):
    try:
        classifier.fit(embeddings, y, sample_weight=sample_weight)
    except TypeError:
        classifier.fit(embeddings, y)
    return classifier


def evaluate_head_candidate(classifier, label_encoder, val_embeddings, val_labels):
    if val_embeddings is None or len(val_labels) == 0:
        return None
    y_val = label_encoder.transform(val_labels)
    y_pred = classifier.predict(val_embeddings)
    labels = list(range(len(label_encoder.classes_)))
    return f1_score(y_val, y_pred, labels=labels, average="macro", zero_division=0)


def fit_task_head(encoder, train_df, task_key, val_df=None):
    task_df = filter_task_dataframe(train_df, task_key, include_synthetic_for_t4=True)
    labels = task_df[TASK_COLUMN[task_key]].apply(normalize_value).tolist()
    embeddings = encode_task_texts(encoder, task_df, task_key)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    sample_weight = np.ones(len(task_df), dtype=float)
    if task_key == "t4":
        synthetic_mask = task_df.apply(is_synthetic_row, axis=1).to_numpy()
        sample_weight[synthetic_mask] = SYNTHETIC_T4_SAMPLE_WEIGHT

    val_embeddings = None
    val_labels = []
    if val_df is not None:
        val_task_df = filter_task_dataframe(val_df, task_key, include_synthetic_for_t4=False)
        val_task_df = val_task_df[
            val_task_df[TASK_COLUMN[task_key]].apply(normalize_value).isin(label_encoder.classes_)
        ]
        if len(val_task_df) > 0:
            val_labels = val_task_df[TASK_COLUMN[task_key]].apply(normalize_value).tolist()
            val_embeddings = encode_task_texts(encoder, val_task_df, task_key)

    best_name = None
    best_clf = None
    best_score = -1.0
    for candidate_name, candidate in build_head_candidates(task_key, y, label_encoder):
        try:
            candidate = fit_with_optional_sample_weight(candidate, embeddings, y, sample_weight)
            selection_score = evaluate_head_candidate(
                candidate,
                label_encoder,
                val_embeddings,
                val_labels,
            )
            if selection_score is None:
                selection_score = f1_score(
                    y,
                    candidate.predict(embeddings),
                    average="macro",
                    zero_division=0,
                )
            print(f"{task_key}: candidate={candidate_name}, selection_macro_f1={selection_score:.4f}")
            if selection_score > best_score:
                best_name = candidate_name
                best_clf = candidate
                best_score = selection_score
        except Exception as exc:
            print(f"{task_key}: candidate={candidate_name} failed: {exc}")

    if best_clf is None:
        raise RuntimeError(f"No valid classifier head for {task_key}")

    print(
        f"{task_key}: selected head={best_name}, score={best_score:.4f}, "
        f"train_rows={len(task_df)}, classes={list(label_encoder.classes_)}"
    )
    return {
        "classifier": best_clf,
        "label_encoder": label_encoder,
        "head_type": best_name,
        "selection_macro_f1": best_score,
    }


def train_fold(fold, seed=SEED):
    set_seed(seed + fold)
    train_df = read_fold_csv(fold, "train")
    val_df = read_fold_csv(fold, "val")
    validate_required_columns(train_df, f"train_fold_{fold}")
    validate_required_columns(val_df, f"val_fold_{fold}")

    encoder = load_encoder()
    encoder = fine_tune_shared_encoder(encoder, train_df, val_df, fold=fold, seed=seed)

    heads = {
        task_key: fit_task_head(encoder, train_df, task_key, val_df=val_df)
        for task_key in ["t1", "t2", "t3", "t4"]
    }

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    encoder.save(str(fold_dir / "encoder"))
    joblib.dump(heads, fold_dir / "task_heads.joblib")

    del encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_dir, val_df


def load_fold_artifacts(fold):
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    encoder = SentenceTransformer(str(fold_dir / "encoder"), device=DEVICE)
    encoder.max_seq_length = MAX_SEQ_LENGTH
    heads = joblib.load(fold_dir / "task_heads.joblib")
    return encoder, heads


In [ ]:
# ==========================================
# 4. Prediction, routing, and metrics
# ==========================================

def predict_task_probabilities(encoder, heads, df, task_key):
    embeddings = encode_task_texts(encoder, df, task_key)
    head = heads[task_key]
    clf = head["classifier"]
    label_encoder = head["label_encoder"]
    raw_probs = clf.predict_proba(embeddings)

    class_probs = pd.DataFrame(0.0, index=df.index, columns=TASK_CLASSES[task_key])
    for class_index, label in enumerate(label_encoder.classes_):
        if label in class_probs.columns:
            encoded_index = int(np.where(label_encoder.classes_ == label)[0][0])
            class_probs[label] = raw_probs[:, encoded_index]
    return class_probs


def predict_all_probabilities_for_fold(fold, df):
    encoder, heads = load_fold_artifacts(fold)
    out = pd.DataFrame({ID_COLUMN: df[ID_COLUMN].values})
    for task_key in ["t1", "t2", "t3", "t4"]:
        probs = predict_task_probabilities(encoder, heads, df.reset_index(drop=True), task_key)
        for cls in TASK_CLASSES[task_key]:
            out[f"{task_key}__{cls}"] = probs[cls].values

    del encoder, heads
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


def average_probability_frames(probability_frames):
    if not probability_frames:
        raise ValueError("No probability frames to average.")
    base = probability_frames[0][[ID_COLUMN]].copy()
    prob_cols = [col for col in probability_frames[0].columns if col != ID_COLUMN]
    for col in prob_cols:
        base[col] = np.mean([frame[col].to_numpy() for frame in probability_frames], axis=0)
    return base


def argmax_from_probs(row, task_key):
    classes = TASK_CLASSES[task_key]
    values = np.array([row[f"{task_key}__{cls}"] for cls in classes], dtype=float)
    return classes[int(values.argmax())]


def route_predictions(prob_df, t1_threshold=0.5, t3_threshold=0.5):
    results = []
    for _, row in prob_df.iterrows():
        t1_yes_prob = float(row["t1__Yes"])
        t3_yes_prob = float(row["t3__Yes"])
        t1_pred = "Yes" if t1_yes_prob >= t1_threshold else "No"

        if t1_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t2_pred = argmax_from_probs(row, "t2")
        t3_pred = "Yes" if t3_yes_prob >= t3_threshold else "No"

        if t3_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2_pred,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t4_pred = argmax_from_probs(row, "t4")
        results.append(
            {
                ID_COLUMN: row[ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2_pred,
                "evidence_status": "Yes",
                "evidence_quality": t4_pred,
            }
        )
    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def strict_task_f1(true_df, pred_df, column, average="macro"):
    merged = true_df[[ID_COLUMN, column]].merge(pred_df[[ID_COLUMN, column]], on=ID_COLUMN, suffixes=("_true", "_pred"))
    y_true = normalize_series(merged[f"{column}_true"])
    y_pred = normalize_series(merged[f"{column}_pred"])
    mask = y_true.notna()
    labels = TASK_CLASSES[{v: k for k, v in TASK_COLUMN.items()}[column]]
    return f1_score(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels, average=average, zero_division=0)


def evaluate_submission(true_df, pred_df):
    rows = []
    for column in TARGET_COLUMNS:
        rows.append(
            {
                "task": column,
                "macro_f1": strict_task_f1(true_df, pred_df, column, average="macro"),
                "weighted_f1": strict_task_f1(true_df, pred_df, column, average="weighted"),
            }
        )
    metrics = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "task": "average",
                "macro_f1": metrics["macro_f1"].mean(),
                "weighted_f1": metrics["weighted_f1"].mean(),
            }
        ]
    )
    competition_score = compute_competition_score(metrics)
    competition = pd.DataFrame(
        [
            {
                "task": "competition",
                "macro_f1": competition_score,
                "weighted_f1": np.nan,
            }
        ]
    )
    return pd.concat([metrics, summary, competition], ignore_index=True)


def compute_competition_score(metrics):
    score = 0.0
    for column, weight in COMPETITION_SCORE_WEIGHTS.items():
        value = float(metrics.loc[metrics["task"] == column, "macro_f1"].iloc[0])
        score += value * weight
    return score


def tune_thresholds(oof_true_df, oof_prob_df):
    best = {"score": -1.0, "objective": "competition_macro_f1", "t1_threshold": 0.5, "t3_threshold": 0.5}
    grid = np.round(np.arange(0.20, 0.801, 0.01), 2)
    for t1_threshold in grid:
        for t3_threshold in grid:
            pred_df = route_predictions(oof_prob_df, t1_threshold=t1_threshold, t3_threshold=t3_threshold)
            metrics = evaluate_submission(oof_true_df, pred_df)
            score = float(metrics.loc[metrics["task"] == "competition", "macro_f1"].iloc[0])
            if score > best["score"]:
                best = {
                    "score": score,
                    "objective": "competition_macro_f1",
                    "t1_threshold": float(t1_threshold),
                    "t3_threshold": float(t3_threshold),
                }
    return best


def print_per_class_reports(true_df, pred_df):
    for column in TARGET_COLUMNS:
        task_key = {v: k for k, v in TASK_COLUMN.items()}[column]
        merged = true_df[[ID_COLUMN, column]].merge(pred_df[[ID_COLUMN, column]], on=ID_COLUMN, suffixes=("_true", "_pred"))
        y_true = normalize_series(merged[f"{column}_true"])
        y_pred = normalize_series(merged[f"{column}_pred"])
        mask = y_true.notna()
        labels = TASK_CLASSES[task_key]
        print(f"\n=== {column} ===")
        print(classification_report(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels, zero_division=0))
        print(pd.DataFrame(confusion_matrix(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels), index=labels, columns=labels))


In [ ]:
# ==========================================
# 5. Train all folds once and create OOF probabilities
# ==========================================

oof_true_frames = []
oof_probability_frames = []
fold_metric_frames = []

for fold in FOLDS:
    print(f"\n{'=' * 48}")
    print(f"Training Fold {fold}")
    print(f"{'=' * 48}")
    _, val_df = train_fold(fold, seed=SEED)
    val_df = val_df.reset_index(drop=True)
    fold_probs = predict_all_probabilities_for_fold(fold, val_df)
    fold_pred = route_predictions(fold_probs, t1_threshold=0.5, t3_threshold=0.5)
    fold_metrics = evaluate_submission(
        val_df[[ID_COLUMN] + TARGET_COLUMNS],
        fold_pred,
    )
    fold_metrics.insert(0, "fold", fold)
    fold_metric_frames.append(fold_metrics)
    display(fold_metrics)
    print_per_class_reports(val_df[[ID_COLUMN] + TARGET_COLUMNS], fold_pred)

    oof_true_frames.append(val_df[[ID_COLUMN] + TARGET_COLUMNS].copy())
    oof_probability_frames.append(fold_probs)

oof_true_df = pd.concat(oof_true_frames, ignore_index=True)
oof_prob_df = pd.concat(oof_probability_frames, ignore_index=True)
oof_prob_df.to_csv(OOF_PROBABILITY_CSV, index=False)
fold_metrics_df = pd.concat(fold_metric_frames, ignore_index=True)
print(f"Saved OOF probabilities to {OOF_PROBABILITY_CSV}")
display(fold_metrics_df)


In [ ]:
# ==========================================
# 6. Threshold tuning, OOF evaluation, and T1/T3 quality gate
# ==========================================

best_thresholds = tune_thresholds(oof_true_df, oof_prob_df)
print("Best thresholds:", best_thresholds)

with open(THRESHOLD_JSON, "w", encoding="utf-8") as f:
    json.dump(best_thresholds, f, ensure_ascii=False, indent=2)

oof_pred_df = route_predictions(
    oof_prob_df,
    t1_threshold=best_thresholds["t1_threshold"],
    t3_threshold=best_thresholds["t3_threshold"],
)
oof_pred_df.to_csv(OOF_PREDICTION_CSV, index=False)
print(f"Saved OOF predictions to {OOF_PREDICTION_CSV}")

oof_metrics = evaluate_submission(oof_true_df, oof_pred_df)
display(oof_metrics)
print_per_class_reports(oof_true_df, oof_pred_df)

quality_gate_rows = []
for task_name, reference_score in REFERENCE_OOF_MACRO_F1.items():
    current_score = float(
        oof_metrics.loc[oof_metrics["task"] == task_name, "macro_f1"].iloc[0]
    )
    drop = reference_score - current_score
    quality_gate_rows.append(
        {
            "task": task_name,
            "reference_macro_f1": reference_score,
            "current_macro_f1": current_score,
            "drop": drop,
            "passed": drop <= T1_T3_MAX_ALLOWED_DROP,
        }
    )
quality_gate_df = pd.DataFrame(quality_gate_rows)
QUALITY_GATE_PASSED = bool(quality_gate_df["passed"].all())
display(quality_gate_df)
if not QUALITY_GATE_PASSED:
    warnings.warn(
        "T1/T3 quality gate failed. Artifacts remain available for diagnosis, "
        "but Hub upload will be blocked."
    )


In [ ]:
# ==========================================
# 7. Baseline comparison against the original final_submission.csv when available
# ==========================================

try:
    baseline_true_df = read_fold_csv(1, "val")[[ID_COLUMN] + TARGET_COLUMNS]
    if LOCAL_FINAL_SUBMISSION.exists():
        baseline_pred_df = pd.read_csv(LOCAL_FINAL_SUBMISSION)[[ID_COLUMN] + TARGET_COLUMNS]
        baseline_metrics = evaluate_submission(baseline_true_df, baseline_pred_df)
        baseline_metrics.insert(0, "model", "original_final_submission")

        setfit_fold1_prob = oof_prob_df[oof_prob_df[ID_COLUMN].isin(baseline_true_df[ID_COLUMN])].copy()
        setfit_fold1_pred = route_predictions(
            setfit_fold1_prob,
            t1_threshold=best_thresholds["t1_threshold"],
            t3_threshold=best_thresholds["t3_threshold"],
        )
        setfit_fold1_metrics = evaluate_submission(baseline_true_df, setfit_fold1_pred)
        setfit_fold1_metrics.insert(0, "model", "setfit_oof_fold_1")
        display(pd.concat([baseline_metrics, setfit_fold1_metrics], ignore_index=True))
    else:
        print(f"Baseline file not found: {LOCAL_FINAL_SUBMISSION}")
except Exception as exc:
    print(f"Baseline comparison skipped: {exc}")


In [ ]:
# ==========================================
# 8. Ensemble inference and export
# ==========================================

def ensemble_inference_and_export(test_csv_path=TEST_CSV_PATH, output_csv_path=SUBMISSION_CSV):
    test_df = pd.read_csv(test_csv_path)
    if ID_COLUMN not in test_df.columns or TEXT_COLUMN not in test_df.columns:
        raise ValueError(f"Test CSV must contain {ID_COLUMN} and {TEXT_COLUMN}.")

    probability_frames = []
    for fold in FOLDS:
        print(f"Predicting with fold {fold}")
        probability_frames.append(predict_all_probabilities_for_fold(fold, test_df.reset_index(drop=True)))

    avg_probs = average_probability_frames(probability_frames)

    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            thresholds = json.load(f)
    else:
        thresholds = {"t1_threshold": 0.5, "t3_threshold": 0.5}

    final_output = route_predictions(
        avg_probs,
        t1_threshold=float(thresholds["t1_threshold"]),
        t3_threshold=float(thresholds["t3_threshold"]),
    )

    final_output[ID_COLUMN] = test_df[ID_COLUMN].values
    final_output = final_output[[ID_COLUMN] + TARGET_COLUMNS]
    final_output.to_csv(output_csv_path, index=False)
    print(f"Exported SetFit submission to {output_csv_path}")
    display(final_output.head())
    return final_output


setfit_submission = ensemble_inference_and_export(TEST_CSV_PATH, SUBMISSION_CSV)


In [ ]:
# ==========================================
# 9. Output format checks
# ==========================================

def validate_output_csv(output_csv_path=SUBMISSION_CSV, test_csv_path=TEST_CSV_PATH):
    output_df = pd.read_csv(output_csv_path)
    test_df = pd.read_csv(test_csv_path)
    expected_columns = [ID_COLUMN] + TARGET_COLUMNS
    assert list(output_df.columns) == expected_columns, f"Unexpected columns: {list(output_df.columns)}"
    assert len(output_df) == len(test_df), f"Row count mismatch: output={len(output_df)}, test={len(test_df)}"
    assert not output_df[ID_COLUMN].duplicated().any(), "Duplicate ids in output."
    assert output_df[ID_COLUMN].tolist() == test_df[ID_COLUMN].tolist(), "Output id order differs from test CSV."
    print("Output CSV format is valid.")
    return output_df


validated_submission = validate_output_csv(SUBMISSION_CSV, TEST_CSV_PATH)


In [ ]:
# ==========================================
# 10. Save SetFit artifact metadata for inference and Hub upload
# ==========================================

def save_setfit_inference_config():
    thresholds = {"t1_threshold": 0.5, "t3_threshold": 0.5}
    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            thresholds.update(json.load(f))

    config = {
        "artifact_version": 2,
        "model_type": "multi_task_setfit",
        "primary_model_name": PRIMARY_MODEL_NAME,
        "fallback_model_name": FALLBACK_MODEL_NAME,
        "max_seq_length": MAX_SEQ_LENGTH,
        "head_ratio": HEAD_RATIO,
        "folds": FOLDS,
        "task_prefix": TASK_PREFIX,
        "task_classes": TASK_CLASSES,
        "target_columns": TARGET_COLUMNS,
        "competition_score_weights": COMPETITION_SCORE_WEIGHTS,
        "encoder_selection": {
            "metric": "weighted_ap",
            "task_weights": {
                "t1": COMPETITION_SCORE_WEIGHTS["promise_status"],
                "t2": COMPETITION_SCORE_WEIGHTS["verification_timeline"],
                "t3": COMPETITION_SCORE_WEIGHTS["evidence_status"],
                "t4": COMPETITION_SCORE_WEIGHTS["evidence_quality"],
            },
            "max_epochs": CONTRASTIVE_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        },
        "pair_sampling": {
            "positive_pair_targets": POSITIVE_PAIR_TARGETS,
            "negative_pairs_per_label": NEGATIVE_PAIRS_PER_LABEL,
            "hard_negative_pair_targets": HARD_NEGATIVE_PAIR_TARGETS,
            "t2_negative_distance_weights": {"1": 3.0, "2": 2.0, "3": 1.0},
            "no_duplicate_pairs": True,
        },
        "head_selection": {
            "default_candidates": [
                "logreg_balanced",
                "logreg_sqrt",
                "linear_svc_calibrated_balanced",
                "linear_svc_calibrated_sqrt",
            ],
            "t2_additional_candidates": [
                "ordinal_logreg_balanced",
                "ordinal_logreg_sqrt",
            ],
            "t4_additional_candidates": ["mlp_256_64"],
            "selection_metric": "fold_validation_macro_f1",
            "synthetic_t4_sample_weight": SYNTHETIC_T4_SAMPLE_WEIGHT,
        },
        "quality_gate": {
            "t1_t3_max_allowed_drop": T1_T3_MAX_ALLOWED_DROP,
            "reference_oof_macro_f1": REFERENCE_OOF_MACRO_F1,
            "passed": globals().get("QUALITY_GATE_PASSED"),
        },
        "thresholds": thresholds,
        "artifact_layout": {
            "encoder": "fold_{fold}/encoder",
            "heads": "fold_{fold}/task_heads.joblib",
            "thresholds": "setfit_thresholds.json",
        },
    }
    with open(INFERENCE_CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print(f"Saved inference config to {INFERENCE_CONFIG_JSON}")
    return config


def write_setfit_model_card(repo_id=None):
    avg_macro = None
    competition_score = None
    try:
        avg_macro = float(oof_metrics.loc[oof_metrics["task"] == "average", "macro_f1"].iloc[0])
        competition_score = float(
            oof_metrics.loc[oof_metrics["task"] == "competition", "macro_f1"].iloc[0]
        )
    except Exception:
        pass

    thresholds = {"t1_threshold": 0.5, "t3_threshold": 0.5}
    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            thresholds.update(json.load(f))

    lines = [
        "# VeriPromiseESG Multi-Task SetFit",
        "",
        "This repository stores SetFit artifacts generated by `app/model/model_setfit.ipynb`.",
        "",
        "## Artifact Layout",
        "",
        "```text",
        "fold_1/encoder/",
        "fold_1/task_heads.joblib",
        "...",
        "fold_5/encoder/",
        "fold_5/task_heads.joblib",
        "setfit_thresholds.json",
        "setfit_inference_config.json",
        "setfit_oof_predictions.csv",
        "setfit_oof_probabilities.csv",
        "```",
        "",
        "## Model",
        "",
        f"- Primary encoder: `{PRIMARY_MODEL_NAME}`",
        f"- Fallback encoder: `{FALLBACK_MODEL_NAME}`",
        "- One shared SentenceTransformer encoder is trained once per fold.",
        "- Encoder checkpoints maximize competition-weighted T1-T4 validation pair AP.",
        "- Task heads are selected by fold validation macro F1.",
        "- T2 supports an ordinal cumulative-logistic head.",
        "- Inference routing matches the original submission format.",
        "",
        "## Thresholds",
        "",
        f"- T1 threshold: `{thresholds.get('t1_threshold', 0.5)}`",
        f"- T3 threshold: `{thresholds.get('t3_threshold', 0.5)}`",
        "",
        "## Validation",
        "",
        f"- OOF average macro F1: `{avg_macro}`" if avg_macro is not None else "- OOF average macro F1: not recorded",
        f"- OOF competition score: `{competition_score}`" if competition_score is not None else "- OOF competition score: not recorded",
        f"- T1/T3 quality gate passed: `{globals().get('QUALITY_GATE_PASSED')}`",
        "",
        "## Inference",
        "",
        "Use `app/model/model_setfit_inference.ipynb` and pass this repository id to `ensemble_inference_and_export`.",
        "",
        f"Repository id: `{repo_id}`" if repo_id else "Repository id: set after upload.",
        "",
        "## Output Columns",
        "",
        "```text",
        "id,promise_status,verification_timeline,evidence_status,evidence_quality",
        "```",
        "",
    ]
    model_card_path = OUTPUT_DIR / "README.md"
    model_card_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved Hugging Face model card to {model_card_path}")
    return model_card_path


setfit_inference_config = save_setfit_inference_config()
write_setfit_model_card(HF_SETFIT_REPO_ID)


In [ ]:
# ==========================================
# 11. Optional: push SetFit artifacts to Hugging Face Hub
# ==========================================
# Set RUN_HF_UPLOAD=True in the configuration cell, then run this cell.
# In Colab, notebook_login() will ask for a Hugging Face token with write access.

def resolve_hf_repo_id(api, repo_id=None):
    if repo_id:
        return repo_id
    username = api.whoami()["name"]
    return f"{username}/VeriPromise_ESG_2026_9906_SetFit"


def validate_setfit_artifacts_for_upload():
    if globals().get("QUALITY_GATE_PASSED") is not True:
        raise RuntimeError("T1/T3 quality gate has not passed; refusing to upload artifacts.")
    missing = []
    for fold in FOLDS:
        fold_dir = OUTPUT_DIR / f"fold_{fold}"
        if not (fold_dir / "encoder").exists():
            missing.append(str(fold_dir / "encoder"))
        if not (fold_dir / "task_heads.joblib").exists():
            missing.append(str(fold_dir / "task_heads.joblib"))
    for path in [THRESHOLD_JSON, INFERENCE_CONFIG_JSON]:
        if not Path(path).exists():
            missing.append(str(path))
    if missing:
        raise FileNotFoundError("Missing SetFit artifacts before upload: " + ", ".join(missing))


def push_setfit_artifacts_to_hf(repo_id=None, private=HF_PRIVATE_REPO):
    save_setfit_inference_config()
    api = HfApi()
    resolved_repo_id = resolve_hf_repo_id(api, repo_id)
    write_setfit_model_card(resolved_repo_id)
    validate_setfit_artifacts_for_upload()

    print(f"Creating or reusing Hugging Face model repo: {resolved_repo_id}")
    api.create_repo(repo_id=resolved_repo_id, repo_type="model", private=private, exist_ok=True)
    print(f"Uploading SetFit artifacts from {OUTPUT_DIR} ...")
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=resolved_repo_id,
        repo_type="model",
        path_in_repo="",
        commit_message=HF_COMMIT_MESSAGE,
    )
    print(f"SetFit artifacts uploaded: https://huggingface.co/{resolved_repo_id}")
    return resolved_repo_id


if RUN_HF_UPLOAD:
    notebook_login()
    uploaded_setfit_repo_id = push_setfit_artifacts_to_hf(HF_SETFIT_REPO_ID, private=HF_PRIVATE_REPO)
else:
    print("RUN_HF_UPLOAD is False. Set it to True after training to upload setfit_outputs to Hugging Face Hub.")
